### Implementing Simple Chatbot using LangGraph

In [ ]:
# Reducers define how updates to a state field are merged with the existing state. By default, updates overwrite previous values.
#  Reducers allow custom merge behavior such as appending messages, combining lists, aggregating results, or maintaining conversation history.
#  The most common reducer is add_messages, which appends new chat messages instead of replacing existing ones.

from typing import Annotated
from langgraph.graph.message import add_messages

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END



In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[str], add_messages]

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
llm.invoke("Hello")



In [ ]:
from langchain_groq import ChatGroq

llm_groq = ChatGroq(model="qwen/qwen3-32b")
llm_groq.invoke("I'm John and like to play basketball")

In [ ]:
### Start of the graph definition

from IPython.display import Image, display

def superbot(state:ChatState):
    return {"messages":[llm_groq.invoke(state["messages"])]}

graph = StateGraph(ChatState)

# Adding nodes to the graph
graph.add_node("superbot", superbot)

# Adding edges to the graph
graph.add_edge(START, "superbot")
graph.add_edge("superbot", END)

graph_builder=graph.compile()

# Display the graph structure
display(Image(graph_builder.get_graph().draw_mermaid_png()))


In [ ]:
# invoke the graph with an initial state
initial_state = {"messages": ["Hello, I'm John and like to play basketball"]}

graph_builder.invoke(initial_state)

In [ ]:
# Streaming responses from the graph
for event in graph_builder.stream(
    {"messages": ["Hello, I am John and like to play basketball"]}, stream_format="json",stream_mode="updates"
):
    print(event)

